# Los axiomas de Zermelo–Fraenkel con Elección (ZFC)

Cuaderno de estudio, en extenso, sobre la teoría axiomática de conjuntos **ZFC**. Para cada axioma se da: enunciado informal, **formalización** en lógica de primer orden, motivación, consecuencias y observaciones; y, cuando es posible, una **ilustración computacional** con conjuntos hereditariamente finitos (HF), es decir, conjuntos construidos sólo con `frozenset`.

> **Advertencia metodológica.** Las celdas de código *ilustran* los axiomas en un modelo finito (HF). No los demuestran: ZFC habla de universos infinitos y los axiomas son **postulados**, no teoremas. Lo que el código sí permite es ver qué afirma cada axioma, construir ejemplos y verificar consecuencias concretas.

## Contenido

0. Marco: lenguaje, primer orden y motivación histórica (paradoja de Russell)
1. Axioma de Extensionalidad
2. Axioma del Conjunto Vacío
3. Axioma del Par
4. Axioma de la Unión
5. Axioma del Conjunto Potencia
6. Axioma de Infinito
7. Esquema axiomático de Separación
8. Esquema axiomático de Reemplazo
9. Axioma de Regularidad (Fundación)
10. Axioma de Elección
11. Resumen, dependencias entre axiomas, variantes (ZF, ZC, Z) e independencia
12. Bibliografía sugerida

## 0. Marco general

### 0.1 El lenguaje

ZFC es una teoría de **primer orden con igualdad** en el lenguaje $\mathcal{L}=\{\in\}$, con un único símbolo no lógico: la relación binaria de pertenencia. No hay otros objetos que conjuntos: los elementos de los conjuntos son, a su vez, conjuntos. Todo lo demás (pares ordenados, relaciones, funciones, números naturales, reales…) se **define** a partir de $\in$.

Abreviaturas usuales (definiciones, no axiomas):

| Abreviatura | Definición&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; |
|---|---|
| $x\subseteq y$ | $\forall z\,(z\in x\to z\in y)$ |
| $x\subsetneq y$ | $x\subseteq y\land x\neq y$ |
| $z=\emptyset$ | $\forall w\,(w\notin z)$ |
| $z=\{x,y\}$ | $\forall w\,(w\in z\leftrightarrow w=x\lor w=y)$ |
| $z=x\cup y$ | $\forall w\,(w\in z\leftrightarrow w\in x\lor w\in y)$ |
| $S(x)=x\cup\{x\}$ | sucesor de $x$ |
| $x$ es *transitivo* | $\forall y\,(y\in x\to y\subseteq x)$ |
| $(\exists!x)\,\varphi(x)$ | $(\exists x)\,\varphi(x)\land\forall y\forall z\,[\varphi(y)\land\varphi(z)\to y=z]$ |

### 0.2 ¿Por qué hacen falta axiomas? La paradoja de Russell

La teoría **ingenua** de Cantor y Frege acepta el principio de comprensión irrestricta:

$$\text{(Comprensión irrestricta)}\qquad \exists y\,\forall x\,\bigl(x\in y\leftrightarrow \varphi(x)\bigr)\quad\text{para toda fórmula }\varphi.$$

Tomando $\varphi(x):\ x\notin x$ y el $y$ que resulta (llamémosle $R$), la instancia $x:=R$ da

$$R\in R\leftrightarrow R\notin R,$$

contradicción (Russell, 1901). Una teoría que prueba una contradicción prueba *todo*, así que hay que **restringir** la formación de conjuntos. Zermelo (1908) propuso una lista de axiomas que construye los conjuntos “desde abajo”, sin conjuntos demasiado grandes; Fraenkel y Skolem (1922) añadieron el Reemplazo y precisaron la noción de propiedad definida (lógica de primer orden); von Neumann introdujo la Regularidad. Con el Axioma de Elección se obtiene **ZFC**.

Como en el capítulo de tablas de verdad, el núcleo proposicional de la paradoja se ve con una tabla: si $p$ es “$R\in R$”, la condición es $p\leftrightarrow\neg p$.

In [1]:
import itertools
import pandas as pd

V = lambda b: 'V' if b else 'F'

# Núcleo proposicional de Russell: p = "R ∈ R".  La definición de R exige  p ↔ ¬p
filas = [{'p = (R∈R)': V(p), '¬p': V(not p), 'p ↔ ¬p': V(p == (not p))} for p in (True, False)]
display(pd.DataFrame(filas))
print('¿Existe asignación consistente con p ↔ ¬p?  ->', any(p == (not p) for p in (True, False)))

,p = (R∈R),¬p,p ↔ ¬p
0,V,F,F
1,F,V,F


¿Existe asignación consistente con p ↔ ¬p?  -> False


Ninguna fila da V: la condición es una **contradicción** (no hay valor de verdad posible para “$R\in R$”). Por tanto, $R$ no puede existir; el axioma de Separación (sec. 7) evita el problema, porque sólo permite “filtrar” elementos **de un conjunto ya dado**.

### 0.3 Herramientas para las ilustraciones

Se representa un conjunto HF como `frozenset` de conjuntos HF. Se define una función de impresión con llaves, y se declaran las construcciones básicas que se usarán en todo el cuaderno.

In [2]:
from itertools import chain, combinations

def fmt(x):
    # Imprime un conjunto HF con llaves y orden canónico (por tamaño y luego texto).
    if not isinstance(x, frozenset):
        return str(x)
    partes = sorted((fmt(y) for y in x), key=lambda s: (len(s), s))
    return '{' + ', '.join(partes) + '}'

def S(*elementos):                     # construye un conjunto HF
    return frozenset(elementos)

VACIO = S()

def potencia(x):
    xs = list(x)
    return frozenset(frozenset(c) for c in chain.from_iterable(combinations(xs, r) for r in range(len(xs) + 1)))

def union_familia(F):
    return frozenset(y for x in F for y in x)

def sucesor(x):
    return x | S(x)                    # S(x) = x ∪ {x}

def von_neumann(n):
    x = VACIO
    for _ in range(n):
        x = sucesor(x)
    return x

print('∅          =', fmt(VACIO))
print('℘(∅)       =', fmt(potencia(VACIO)))
print('℘({∅,{∅}})=', fmt(potencia(S(VACIO, S(VACIO)))))

∅          = {}
℘(∅)       = {{}}
℘({∅,{∅}})= {{}, {{}}, {{{}}}, {{}, {{}}}}


## 1. Axioma de Extensionalidad

**Enunciado informal.** Dos conjuntos con los mismos elementos son iguales.

**Formalización:**

$$\forall x\,\forall y\,\bigl[\forall z\,(z\in x\leftrightarrow z\in y)\to x=y\bigr]$$

(La implicación inversa, $x=y\to\forall z\,(z\in x\leftrightarrow z\in y)$, es consecuencia de los axiomas de la igualdad.)

**Significado y consecuencias.**
- Un conjunto queda **determinado por sus elementos**, no por la forma de describirlo, su orden ni su multiplicidad: $\{1,2\}=\{2,1\}=\{1,1,2\}$.
- Es el único axioma que “habla” de la igualdad; justifica la **técnica de demostración por doble inclusión**: $x=y\iff x\subseteq y\land y\subseteq x$.
- Sin Extensionalidad se tendrían “átomos” o entes con distinta identidad aunque tengan los mismos elementos (p. ej., dos conjuntos vacíos distintos). De hecho Extensionalidad + Conjunto Vacío dan la **unicidad** del vacío (sec. 2).
- Da sentido al símbolo $\{x: \varphi(x)\}$: si existe un conjunto con esos elementos, es único.

**Ilustración.** En Python, `frozenset` ya es extensional: sólo importan los elementos.

In [3]:
a = S(1, 2)
b = S(2, 1)
c = frozenset([1, 1, 2, 2])
print('{1,2} = {2,1}      :', a == b)
print('{1,2} = {1,1,2,2}  :', a == c)

# Demostración por doble inclusión
def doble_inclusion(x, y):
    return x <= y and y <= x
print('doble inclusión ⇔ igualdad:', doble_inclusion(a, b) == (a == b))

# Extensionalidad en universos HF de 2 niveles: dos conjuntos con mismos elementos coinciden siempre
U = [VACIO, S(VACIO), S(S(VACIO)), S(VACIO, S(VACIO))]
todos = [frozenset(c) for r in range(len(U) + 1) for c in combinations(U, r)]
ok = all((x == y) == all((z in x) == (z in y) for z in U) for x in todos for y in todos)
print('Extensionalidad verificada en', len(todos), 'conjuntos (', len(todos)**2, 'pares ):', ok)

{1,2} = {2,1}      : True
{1,2} = {1,1,2,2}  : True
doble inclusión ⇔ igualdad: True
Extensionalidad verificada en 16 conjuntos ( 256 pares ): True


## 2. Axioma del Conjunto Vacío (existencia)

**Enunciado informal.** Existe un conjunto sin elementos.

**Formalización:**

$$\exists x\,\forall y\,(y\notin x)$$

**Observaciones.**
- En la lógica de primer orden el universo del discurso es **no vacío**, por lo que $\exists x\,(x=x)$ es un teorema lógico y el Conjunto Vacío se deduce del esquema de Separación (sec. 7): si $a$ es cualquier conjunto, $\{x\in a: x\neq x\}$ es vacío. Muchos textos lo enuncian igualmente como axioma por claridad (y porque Z lo tomó como postulado).
- **Unicidad.** Si $\emptyset_1,\emptyset_2$ son vacíos, tienen los mismos elementos (ninguno), así que por Extensionalidad $\emptyset_1=\emptyset_2$. Se escribe $\emptyset$.
- El vacío es el “ladrillo cero”: con él y las operaciones de los demás axiomas se construye toda la jerarquía: $\emptyset,\ \{\emptyset\},\ \{\{\emptyset\}\},\ \{\emptyset,\{\emptyset\}\},\dots$
- $\emptyset\subseteq x$ para todo $x$ (verdad vacua: no hay elemento de $\emptyset$ fuera de $x$).

In [4]:
print('∅ =', fmt(VACIO), '| |∅| =', len(VACIO))
print('∅ ⊆ x para todo x de prueba:', all(VACIO <= x for x in [VACIO, S(VACIO), S(S(VACIO)), S(1, 2, 3)]))
print('¬(∅ ∈ ∅):', VACIO not in VACIO,  '   pero   ∅ ∈ {∅}:', VACIO in S(VACIO))
# Distinción entre ∅ y {∅}: no son iguales (uno tiene elementos y el otro no)
print('∅ = {∅}?', VACIO == S(VACIO))

∅ = {} | |∅| = 0
∅ ⊆ x para todo x de prueba: True
¬(∅ ∈ ∅): True    pero   ∅ ∈ {∅}: True
∅ = {∅}? False


## 3. Axioma del Par

**Enunciado informal.** Dados dos conjuntos $a$ y $b$, existe un conjunto cuyos únicos elementos son $a$ y $b$.

**Formalización:**

$$\forall a\,\forall b\,\exists c\,\forall x\,\bigl(x\in c\leftrightarrow x=a\lor x=b\bigr)$$

Por Extensionalidad el conjunto $c$ es único; se escribe $\{a,b\}$. Si $a=b$ se obtiene el **singleton** $\{a\}=\{a,a\}$.

**Consecuencias.**
- Con el vacío: $\{\emptyset\}$, $\{\{\emptyset\}\}$, $\{\emptyset,\{\emptyset\}\}$, … (infinitos conjuntos, aunque *no* un conjunto infinito todavía).
- **Par ordenado de Kuratowski:**
$$(a,b):=\{\{a\},\{a,b\}\}$$
que cumple la propiedad fundamental $(a,b)=(c,d)\iff a=c\land b=d$. Aquí es donde el Par permite definir **relaciones y funciones** como conjuntos de pares ordenados (junto con Unión, Potencia y Separación para el producto cartesiano).
- El Par (con Unión) permite formar conjuntos finitos por enumeración: $\{a_1,\dots,a_n\}=\{a_1\}\cup\cdots\cup\{a_n\}$.

**Ilustración:** se verifica la propiedad del par ordenado sobre un universo pequeño.

In [5]:
def par(a, b):      return S(a, b)
def singleton(a):   return S(a)
def par_ord(a, b):  return S(singleton(a), par(a, b))    # Kuratowski

print('{a,b} con a=b es el singleton:', par(1, 1) == singleton(1))
print('(1,2) =', fmt(par_ord(1, 2)), '   (2,1) =', fmt(par_ord(2, 1)), '   (1,1) =', fmt(par_ord(1, 1)))

D = [0, 1, 2, 3]
ok = all((par_ord(a, b) == par_ord(c, d)) == (a == c and b == d)
         for a, b, c, d in itertools.product(D, repeat=4))
print('(a,b)=(c,d) ⇔ a=c ∧ b=d verificado en', len(D)**4, 'casos:', ok)

{a,b} con a=b es el singleton: True
(1,2) = {{1}, {1, 2}}    (2,1) = {{2}, {1, 2}}    (1,1) = {{1}}
(a,b)=(c,d) ⇔ a=c ∧ b=d verificado en 256 casos: True


## 4. Axioma de la Unión

**Enunciado informal.** Para toda familia $\mathcal F$ de conjuntos existe un conjunto formado exactamente por los elementos de los elementos de $\mathcal F$.

**Formalización:**

$$\forall\mathcal F\,\exists u\,\forall x\,\bigl(x\in u\leftrightarrow\exists y\,(y\in\mathcal F\land x\in y)\bigr)$$

El conjunto $u$ es único (Extensionalidad) y se escribe $\bigcup\mathcal F$.

**Consecuencias.**
- La unión binaria: $a\cup b:=\bigcup\{a,b\}$ (se usa el Par).
- $\bigcup\emptyset=\emptyset$ y $\bigcup\{a\}=a$.
- Para el sucesor: $S(x)=x\cup\{x\}$ existe por Par + Unión.
- Las **intersecciones** no requieren axioma propio (se obtienen por Separación): para $\mathcal F\neq\emptyset$, $\bigcap\mathcal F=\{x\in\bigcup\mathcal F:\forall y\in\mathcal F\,(x\in y)\}$. Para $\mathcal F=\emptyset$ la intersección “sería el universo”, que **no es un conjunto**: por eso se exige $\mathcal F\neq\emptyset$.
- Leyes: $\bigcup(\mathcal F\cup\mathcal G)=\bigcup\mathcal F\cup\bigcup\mathcal G$, y si $\mathcal F\subseteq\mathcal G$ entonces $\bigcup\mathcal F\subseteq\bigcup\mathcal G$.

In [6]:
F = S(S(1, 2), S(2, 3), S(4))
print('F       =', fmt(F))
print('⋃F      =', fmt(union_familia(F)))
print('⋃∅      =', fmt(union_familia(VACIO)))
print('⋃{a}=a  :', union_familia(S(S(7, 8))) == S(7, 8))

def interseccion_familia(F):
    # Se define por Separación a partir de ⋃F; exige F no vacía
    if not F:
        raise ValueError('⋂∅ no es un conjunto (sería el universo)')
    U = union_familia(F)
    return frozenset(x for x in U if all(x in y for y in F))

print('⋂{{1,2},{2,3}} =', fmt(interseccion_familia(S(S(1, 2), S(2, 3)))))
try:
    interseccion_familia(VACIO)
except ValueError as e:
    print('⋂∅ ->', e)

F       = {{4}, {1, 2}, {2, 3}}
⋃F      = {1, 2, 3, 4}
⋃∅      = {}
⋃{a}=a  : True
⋂{{1,2},{2,3}} = {2}
⋂∅ -> ⋂∅ no es un conjunto (sería el universo)


## 5. Axioma del Conjunto Potencia

**Enunciado informal.** Para todo conjunto $a$ existe el conjunto cuyos elementos son exactamente los subconjuntos de $a$.

**Formalización:**

$$\forall a\,\exists p\,\forall x\,\bigl(x\in p\leftrightarrow x\subseteq a\bigr)$$

Se escribe $\mathcal P(a)$.

**Consecuencias.**
- $|\mathcal P(a)|=2^{|a|}$ para $a$ finito.
- Junto con Par, Unión y Separación permite definir el **producto cartesiano** $a\times b\subseteq\mathcal P(\mathcal P(a\cup b))$ (pues $(x,y)=\{\{x\},\{x,y\}\}$), y por tanto **relaciones, funciones**, el conjunto ${}^{b}a$ de funciones de $b$ en $a$, etc.
- **Teorema de Cantor:** no existe sobreyección $a\to\mathcal P(a)$; luego $|a|<|\mathcal P(a)|$ para *todo* $a$. Consecuencia: existen infinitos de tamaños distintos, y **no existe el conjunto de todos los conjuntos** (si $V$ existiera, $\mathcal P(V)\subseteq V$ contradiría Cantor).
- Es el axioma que da la *enorme* riqueza de ZFC: es responsable de que el universo crezca tanto que ciertas preguntas (como la Hipótesis del Continuo, $2^{\aleph_0}=\aleph_1$) sean **independientes** (sec. 11).
- Es un axioma “impredicativo”: $\mathcal P(a)$ se define cuantificando sobre **todos** los subconjuntos posibles de $a$.

**Ilustración:** potencia, conteo $2^n$ y verificación exhaustiva del teorema de Cantor para $|a|=3$.

In [7]:
a = S(1, 2, 3)
print('℘({1,2,3}) =', fmt(potencia(a)))
print('|℘(a)| =', len(potencia(a)), ' = 2^|a| =', 2**len(a))

# Teorema de Cantor, verificación exhaustiva: ninguna f: a → ℘(a) es sobreyectiva
elems = sorted(a)
P = list(potencia(a))
n_funciones = 0
sobreyectivas = 0
for imagenes in itertools.product(P, repeat=len(elems)):        # 8^3 = 512 funciones
    n_funciones += 1
    if set(imagenes) == set(P):
        sobreyectivas += 1
print(f'funciones a→℘(a): {n_funciones};  sobreyectivas: {sobreyectivas}')

# La demostración de Cantor: el conjunto diagonal D = {x ∈ a : x ∉ f(x)} nunca está en la imagen
ejemplo = dict(zip(elems, [S(1, 2), S(), S(1, 3)]))
D = frozenset(x for x in elems if x not in ejemplo[x])
print('f =', {k: fmt(v) for k, v in ejemplo.items()}, ' D =', fmt(D), '| D en la imagen de f:', D in ejemplo.values())

℘({1,2,3}) = {{}, {1}, {2}, {3}, {1, 2}, {1, 3}, {2, 3}, {1, 2, 3}}
|℘(a)| = 8  = 2^|a| = 8
funciones a→℘(a): 512;  sobreyectivas: 0
f = {1: '{1, 2}', 2: '{}', 3: '{1, 3}'}  D = {2} | D en la imagen de f: False


## 6. Axioma de Infinito

**Enunciado informal.** Existe un conjunto que contiene al vacío y es cerrado bajo el sucesor $x\mapsto x\cup\{x\}$ (un conjunto *inductivo*).

**Formalización:**

$$\exists I\,\bigl[\emptyset\in I\ \land\ \forall x\,(x\in I\to x\cup\{x\}\in I)\bigr]$$

**Por qué es necesario.** Todos los axiomas anteriores *conservan* la finitud: a partir de conjuntos finitos sólo producen conjuntos finitos. Sin un axioma explícito no habría ningún conjunto infinito (los conjuntos HF son un modelo de todo ZFC salvo Infinito).

**Construcción de $\omega$ (los naturales de von Neumann).** Con los números naturales definidos como

$$0=\emptyset,\quad 1=\{0\},\quad 2=\{0,1\},\quad \dots,\quad n+1=n\cup\{n\}=\{0,1,\dots,n\},$$

se define $\omega$ como la **intersección de todos los conjuntos inductivos** (un subconjunto de cualquier $I$ dado, por Separación):

$$\omega=\bigcap\{J\subseteq I: J\text{ inductivo}\}=\{n\in I:\ n \text{ pertenece a todo subconjunto inductivo de }I\}$$

Es el menor conjunto inductivo, y de ello se sigue el **principio de inducción**: si $A\subseteq\omega$ es inductivo, $A=\omega$. Sobre $\omega$ se definen $+,\cdot,<$ (con $<$ siendo $\in$) y se prueban los axiomas de Peano. Así, **toda la aritmética** se interpreta en ZFC.

**Propiedades de los naturales de von Neumann:** cada $n$ es un conjunto **transitivo**; $m<n\iff m\in n$; $|n|=n$ (tiene exactamente $n$ elementos); $\in$ ordena totalmente a cada $n$.

**Ilustración:** se construyen los primeros naturales y se verifican esas propiedades.

In [8]:
N = [von_neumann(n) for n in range(6)]
for n, x in enumerate(N):
    print(f'{n} =', fmt(x))

def transitivo(x):  return all(y <= x for y in x)
print('\nCada n es transitivo          :', all(transitivo(x) for x in N))
print('|n| = n                       :', all(len(x) == n for n, x in enumerate(N)))
print('m<n ⇔ m∈n (m,n<6)            :', all((m < n) == (N[m] in N[n]) for m in range(6) for n in range(6)))
print('m<n ⇔ m⊊n  (inclusión propia):', all((m < n) == (N[m] < N[n]) for m in range(6) for n in range(6)))
print('n+1 = n ∪ {n}                 :', all(N[n + 1] == N[n] | S(N[n]) for n in range(5)))

# Un fragmento de conjunto inductivo: {0,1,2,3,4,5} NO es inductivo, por eso se necesita un axioma
Ifin = frozenset(N)
inductivo = VACIO in Ifin and all(sucesor(x) in Ifin for x in Ifin)
print('\n¿{0,1,...,5} es inductivo? ->', inductivo, '(falla: S(5)=6 ∉ conjunto)')

0 = {}
1 = {{}}
2 = {{}, {{}}}
3 = {{}, {{}}, {{}, {{}}}}
4 = {{}, {{}}, {{}, {{}}}, {{}, {{}}, {{}, {{}}}}}
5 = {{}, {{}}, {{}, {{}}}, {{}, {{}}, {{}, {{}}}}, {{}, {{}}, {{}, {{}}}, {{}, {{}}, {{}, {{}}}}}}

Cada n es transitivo          : True
|n| = n                       : True
m<n ⇔ m∈n (m,n<6)            : True
m<n ⇔ m⊊n  (inclusión propia): True
n+1 = n ∪ {n}                 : True

¿{0,1,...,5} es inductivo? -> False (falla: S(5)=6 ∉ conjunto)


**Observación.** No existe en HF un conjunto inductivo: *en cualquier modelo finito* el sucesor de un elemento “máximo” saldría del modelo. El Axioma de Infinito afirma que el universo **no** es de este tipo. Con él existen, además de $\omega$, los conjuntos $\omega+1=\omega\cup\{\omega\}$, $\omega+2$, … y (con Reemplazo) $\omega\cdot 2$, $\omega^2$, $\varepsilon_0$, etc.: los **ordinales**.

## 7. Esquema axiomático de Separación (Aussonderung / Comprensión restringida)

**Enunciado informal.** Dado un conjunto $a$ y una propiedad $\varphi$, existe el subconjunto de $a$ formado por los elementos que cumplen $\varphi$.

**Formalización.** Es un **esquema**: una instancia por cada fórmula $\varphi(x,p_1,\dots,p_n)$ en la que $b$ **no** ocurra libre:

$$\forall p_1\cdots\forall p_n\,\forall a\,\exists b\,\forall x\,\bigl(x\in b\leftrightarrow x\in a\land\varphi(x,p_1,\dots,p_n)\bigr)$$

Se escribe $b=\{x\in a:\varphi(x)\}$.

**Por qué es un esquema y no un axioma.** La lógica de primer orden no puede cuantificar sobre “propiedades”; se necesita una cláusula para *cada* fórmula. En consecuencia, ZFC tiene **infinitos** axiomas (no es finitamente axiomatizable).

**Diferencia clave con la comprensión irrestricta.** $\varphi$ sólo filtra dentro de un conjunto **ya existente** $a$. Esto bloquea Russell:

> **Teorema (no hay conjunto universal).** Para todo conjunto $a$ existe un $R\notin a$.
>
> *Demostración.* Sea $R=\{x\in a: x\notin x\}$. Si $R\in a$: por definición $R\in R\iff R\in a\land R\notin R\iff R\notin R$ (pues $R\in a$), contradicción. Por tanto $R\notin a$. $\blacksquare$

En consecuencia **no existe $V=\{x:x=x\}$**: sería un $a$ con $R\in a$. Las “colecciones demasiado grandes” (la clase de todos los conjuntos, de todos los ordinales…) son **clases propias**.

**Consecuencias.**
- Existencia de $a\cap b=\{x\in a:x\in b\}$, $a\setminus b=\{x\in a:x\notin b\}$, y de $\bigcap\mathcal F$.
- Deriva el Conjunto Vacío: $\{x\in a: x\neq x\}$.
- Junto a Potencia: existe $\{x\in\mathcal P(a):\varphi(x)\}$, y de ahí los conjuntos de funciones y el producto cartesiano.
- **Clases.** Una “clase” $\{x:\varphi(x)\}$ es un mero abuso de notación para la fórmula $\varphi$; es un conjunto sólo si hay un $a$ que la contiene (Separación).

**Ilustración:** filtrar, y verificar el teorema de no-universalidad sobre universos finitos.

In [9]:
def separacion(a, phi):
    return frozenset(x for x in a if phi(x))

a = frozenset(range(1, 11))
print('{x∈{1..10} : x par}      =', sorted(separacion(a, lambda x: x % 2 == 0)))
print('{x∈{1..10} : x primo}    =', sorted(separacion(a, lambda x: x > 1 and all(x % d for d in range(2, x)))))
print('Vacío por separación      =', fmt(separacion(a, lambda x: x != x)))

# Teorema: para todo a, R = {x∈a : x∉x} no pertenece a a
def russell(a):
    return separacion(a, lambda x: x not in x if isinstance(x, frozenset) else True)

prueba = [VACIO, S(VACIO), S(VACIO, S(VACIO)), von_neumann(3), S(S(S(VACIO)), VACIO)]
for x in prueba:
    R = russell(x)
    print(f'a = {fmt(x):<34}  R = {fmt(R):<34}  R ∈ a ? {R in x}')

{x∈{1..10} : x par}      = [2, 4, 6, 8, 10]
{x∈{1..10} : x primo}    = [2, 3, 5, 7]
Vacío por separación      = {}
a = {}                                  R = {}                                  R ∈ a ? False
a = {{}}                                R = {{}}                                R ∈ a ? False
a = {{}, {{}}}                          R = {{}, {{}}}                          R ∈ a ? False
a = {{}, {{}}, {{}, {{}}}}              R = {{}, {{}}, {{}, {{}}}}              R ∈ a ? False
a = {{}, {{{}}}}                        R = {{}, {{{}}}}                        R ∈ a ? False


## 8. Esquema axiomático de Reemplazo (Fraenkel, Skolem)

**Enunciado informal.** La imagen de un conjunto bajo una “función definible” es un conjunto.

**Formalización.** Sea $\varphi(x,y,p_1,\dots,p_n)$ una fórmula. Si $\varphi$ define una función (funcional) sobre un conjunto $a$, entonces existe el conjunto de las imágenes:

$$\forall p\;\forall a\,\Bigl[\bigl(\forall x\in a\;\exists!\,y\;\varphi(x,y,p)\bigr)\to\exists b\;\forall y\,\bigl(y\in b\leftrightarrow\exists x\in a\;\varphi(x,y,p)\bigr)\Bigr]$$

Se escribe $b=\{F(x):x\in a\}$. Al igual que Separación, es un **esquema** (una cláusula por fórmula).

**Por qué es necesario.** Sin Reemplazo no pueden demostrarse, por ejemplo:
- la existencia del conjunto $\{\omega,\mathcal P(\omega),\mathcal P(\mathcal P(\omega)),\dots\}$;
- que todo conjunto bien ordenado es isomorfo a un ordinal de von Neumann (teorema de representación), y por tanto la existencia de $\omega+\omega$ o de $\aleph_\omega$ como conjuntos;
- el **teorema de recursión transfinita**.
Zermelo (Z, sin Reemplazo) tiene un modelo ($V_{\omega+\omega}$) en el que los $\beth_n$ existen pero el conjunto $\{\beth_n:n<\omega\}$ no.

**Consecuencias.**
- **Implica Separación.** Dada $\varphi$ y $a$, se toma $F(x)=x$ si $\varphi(x)$ y $F(x)=x_0$ para un $x_0\in a$ fijo con $\varphi(x_0)$ en caso contrario (funciona si existe $x_0$; si no, el resultado es $\emptyset$). Por eso, en algunas presentaciones, Separación se omite como axioma independiente.
- Permite **construir ordinales** grandes y definir funciones por recursión sobre ordinales; es la base del cálculo transfinito.
- Extiende la **jerarquía acumulativa** $V_0=\emptyset,\ V_{\alpha+1}=\mathcal P(V_\alpha),\ V_\lambda=\bigcup_{\alpha<\lambda}V_\alpha$ a todos los ordinales.
- Es un axioma “de tamaño”: los conjuntos “pequeños” (tamaño $\le$ un conjunto dado) se preservan. Equivale a decir que la **clase $V$ no puede escribirse como imagen de un conjunto**.

**Ilustración:** se aplica $F(n)=V_n$ (iteración de la potencia) sobre $\{0,\dots,4\}$ y se cuenta la imagen.

In [10]:
def reemplazo(a, F):
    return frozenset(F(x) for x in a)

def V_n(n):                      # jerarquía acumulativa: V_0=∅, V_{n+1}=℘(V_n)
    x = VACIO
    for _ in range(n):
        x = potencia(x)
    return x

a = range(5)
imagen = reemplazo(a, V_n)
print('{V_n : n<5} tiene', len(imagen), 'elementos (los V_n son distintos entre sí)')
print('|V_n| para n=0..4 :', [len(V_n(n)) for n in range(5)], '  (0, 1, 2, 4, 16)')
print('|V_5| = 2^16 =', 2**16, ' (se calcula: ', len(V_n(5)), ')')

# Reemplazo no inyectivo: F constante da un singleton; la imagen nunca supera al dominio
print('Imagen de {0,..,4} bajo F(x)=∅ :', fmt(reemplazo(a, lambda x: VACIO)))
print('|imagen| ≤ |dominio| siempre   :', all(len(reemplazo(a, f)) <= len(a) for f in (lambda x: x % 2, lambda x: x, lambda x: 0)))

{V_n : n<5} tiene 5 elementos (los V_n son distintos entre sí)
|V_n| para n=0..4 : [0, 1, 2, 4, 16]   (0, 1, 2, 4, 16)


|V_5| = 2^16 = 65536  (se calcula:  65536 )
Imagen de {0,..,4} bajo F(x)=∅ : {{}}
|imagen| ≤ |dominio| siempre   : True


## 9. Axioma de Regularidad (Fundación)

**Enunciado informal.** Todo conjunto no vacío tiene un elemento $\in$-minimal: un elemento con el que no comparte ningún elemento.

**Formalización:**

$$\forall x\,\bigl[x\neq\emptyset\to\exists y\,(y\in x\land y\cap x=\emptyset)\bigr]$$

(equivalentemente: $\forall x\,[\exists y\,(y\in x)\to\exists y\in x\,\forall z\,(z\in y\to z\notin x)]$).

**Consecuencias.**
- **Nadie se pertenece a sí mismo:** $x\notin x$ para todo $x$. (Aplicando el axioma a $\{x\}$: su único elemento $x$ debe cumplir $x\cap\{x\}=\emptyset$, es decir, $x\notin x$.)
- **No hay ciclos de pertenencia** $x\in y\in x$: si los hubiera, $\{x,y\}$ violaría la regularidad. Tampoco hay **sucesiones descendentes infinitas** $\dots\in x_2\in x_1\in x_0$: el conjunto $\{x_n:n<\omega\}$ no tendría elemento $\in$-minimal (esto se demuestra ya en ZF; el recíproco sí requiere algo de elección, p. ej. Elección Dependiente).
- **Inducción bien fundada ($\in$-inducción):** si una propiedad pasa de todos los elementos de $x$ a $x$, vale para todo conjunto.
- **Rango.** Todo conjunto pertenece a algún $V_\alpha$: el universo es la unión de la jerarquía acumulativa, $V=\bigcup_{\alpha\in\mathrm{Ord}}V_\alpha$. El **rango** de $x$ es el menor $\alpha$ con $x\in V_{\alpha+1}$. (Este resultado es de hecho *equivalente* a Regularidad en presencia de los demás axiomas.)
- Sólo **restringe el universo** a los conjuntos “bien fundados”; no es necesaria para el desarrollo habitual de las matemáticas (*ZFC menos Regularidad* ya las contiene), pero da una imagen clara y elimina patologías como $x=\{x\}$ (conjuntos de Quine).

**Ilustración:** se calcula el rango y se comprueba que en HF $x\notin x$ y que todo conjunto no vacío tiene un elemento minimal.

In [11]:
def rango(x):
    # rk(∅)=0, rk(x)=sup{rk(y)+1 : y∈x}
    return 0 if not x else max(rango(y) + 1 for y in x)

def minimal(x):
    # elemento y de x con y ∩ x = ∅   (existe por Regularidad)
    for y in x:
        if not (y & x):
            return y
    return None

muestra = [von_neumann(n) for n in range(6)] + [S(VACIO, S(S(VACIO))), potencia(von_neumann(3)), S(S(VACIO), S(S(VACIO)))]
for x in muestra:
    print(f'x = {fmt(x):<44} rango = {rango(x)}   minimal = {fmt(minimal(x)) if x else "—"}')

print('\nNingún x de la muestra cumple x∈x     :', all(x not in x for x in muestra))
print('Todo x ≠ ∅ tiene elemento minimal      :', all(minimal(x) is not None for x in muestra if x))
print('rango(n) = n  (naturales de von Neumann):', all(rango(von_neumann(n)) == n for n in range(8)))

# Una "violación" imaginaria: a = {a} no se puede construir con frozenset (por eso HF ⊨ Regularidad)

x = {}                                           rango = 0   minimal = —
x = {{}}                                         rango = 1   minimal = {}
x = {{}, {{}}}                                   rango = 2   minimal = {}
x = {{}, {{}}, {{}, {{}}}}                       rango = 3   minimal = {}
x = {{}, {{}}, {{}, {{}}}, {{}, {{}}, {{}, {{}}}}} rango = 4   minimal = {}
x = {{}, {{}}, {{}, {{}}}, {{}, {{}}, {{}, {{}}}}, {{}, {{}}, {{}, {{}}}, {{}, {{}}, {{}, {{}}}}}} rango = 5   minimal = {}
x = {{}, {{{}}}}                                 rango = 3   minimal = {}
x = {{}, {{}}, {{{}}}, {{}, {{}}}, {{{}, {{}}}}, {{}, {{}, {{}}}}, {{{}}, {{}, {{}}}}, {{}, {{}}, {{}, {{}}}}} rango = 4   minimal = {}
x = {{{}}, {{{}}}}                               rango = 3   minimal = {{}}

Ningún x de la muestra cumple x∈x     : True
Todo x ≠ ∅ tiene elemento minimal      : True
rango(n) = n  (naturales de von Neumann): True


## 10. Axioma de Elección (AC)

**Enunciado informal.** Toda familia de conjuntos no vacíos **disjuntos dos a dos** admite un conjunto que contiene exactamente un elemento de cada uno (un *conjunto de selección*).

**Formalización (versión con conjuntos de selección):**

$$\forall\mathcal F\,\Bigl[\bigl(\emptyset\notin\mathcal F\land\forall x,y\in\mathcal F\,(x\neq y\to x\cap y=\emptyset)\bigr)\to\exists c\;\forall x\in\mathcal F\;\exists!\,z\,(z\in x\cap c)\Bigr]$$

**Formulación equivalente (función de elección).** Para todo conjunto $\mathcal F$ con $\emptyset\notin\mathcal F$ existe una función $f:\mathcal F\to\bigcup\mathcal F$ con $f(x)\in x$ para todo $x\in\mathcal F$.

**Equivalentes famosos** (en ZF):
- **Lema de Zorn:** un orden parcial donde toda cadena tiene cota superior posee un elemento maximal.
- **Teorema del Buen Orden (Zermelo):** todo conjunto se puede bien ordenar.
- **Tricotomía cardinal:** para cualesquiera $A,B$, $|A|\le|B|$ o $|B|\le|A|$.
- **Producto cartesiano:** el producto de una familia de conjuntos no vacíos es no vacío.
- **Lema de Tychonoff** (producto de compactos es compacto) y **todo espacio vectorial tiene base** (Hamel).

**Consecuencias “sorprendentes”.** Existencia de conjuntos no medibles de Lebesgue (Vitali); **paradoja de Banach–Tarski** (descomposición de una esfera en piezas reensamblables en dos esferas idénticas); existencia de ultrafiltros no principales. Por ello AC fue **polémico** (Borel, Lebesgue, Baire) antes de aceptarse de forma general.

**Independencia (Gödel 1938, Cohen 1963).** AC es consistente con ZF (Gödel, mediante el universo construible $L$) y su negación también es consistente con ZF (Cohen, mediante *forcing*). Es decir, **ZF no decide AC**.

**Finitud vs. infinitud.** Para familias *finitas* AC es **teorema** de ZF (por inducción); para infinitas se necesita el axioma, salvo que exista una regla **explícita** (ejemplo: elegir el menor elemento de subconjuntos no vacíos de $\mathbb N$, o la regla de Russell “un zapato de cada par”, frente a “un calcetín de cada par” que exige AC).

**Ilustración:** función de elección para una familia finita, conjunto de selección y verificación de que el producto de conjuntos no vacíos es no vacío.

In [12]:
# Familia finita de conjuntos no vacíos y disjuntos
F = [frozenset({1, 2, 3}), frozenset({4, 5}), frozenset({6}), frozenset({7, 8, 9, 10})]

# Regla explícita (sin necesitar AC): elegir el mínimo
f = {x: min(x) for x in F}
print('función de elección  f(x)=min x :', {tuple(sorted(x)): v for x, v in f.items()})

# Conjunto de selección c: un único elemento de cada miembro de la familia
c = frozenset(f.values())
print('conjunto de selección c          =', sorted(c))
print('|c ∩ x| = 1 para todo x∈F       :', all(len(c & x) == 1 for x in F))

# Equivalente "producto no vacío": el producto cartesiano de conjuntos no vacíos es no vacío
producto = list(itertools.product(*[sorted(x) for x in F]))
print('|∏ F| =', len(producto), '(=3·2·1·4)   no vacío:', len(producto) > 0, '  —un elemento del producto es una función de elección')
print('primer elemento del producto     =', producto[0])

# Si hubiera un conjunto vacío en la familia, no habría función de elección ni producto
print('familia con ∅: ¿producto vacío?  ', len(list(itertools.product(*[sorted(x) for x in F + [frozenset()]]))) == 0)

función de elección  f(x)=min x : {(1, 2, 3): 1, (4, 5): 4, (6,): 6, (7, 8, 9, 10): 7}
conjunto de selección c          = [1, 4, 6, 7]
|c ∩ x| = 1 para todo x∈F       : True
|∏ F| = 24 (=3·2·1·4)   no vacío: True   —un elemento del producto es una función de elección
primer elemento del producto     = (1, 4, 6, 7)
familia con ∅: ¿producto vacío?   True


**Comentario.** Para familias finitas la selección se hace explícitamente por inducción; la **necesidad** de AC aparece sólo cuando hay *infinitas* elecciones sin regla definible. En código Python esto no se puede ilustrar (todo objeto computable viene dado por una regla), lo cual está en sintonía con la idea: *AC es un axioma no constructivo*.

## 11. Resumen, dependencias y metateoría

### 11.1 Tabla de los axiomas de ZFC

In [13]:
filas = [
 ('1', 'Extensionalidad', 'Igualdad de conjuntos',       '∀x∀y[∀z(z∈x↔z∈y) → x=y]', 'axioma', 'Unicidad de ∅, {a,b}, ⋃F, ℘(a)…'),
 ('2', 'Conjunto Vacío',  'Existencia de ∅',              '∃x∀y(y∉x)', 'axioma (deducible de Separación)', 'Base de la jerarquía'),
 ('3', 'Par',             'Formación de {a,b}',           '∀a∀b∃c∀x(x∈c↔x=a∨x=b)', 'axioma', 'Par ordenado, singleton'),
 ('4', 'Unión',           'Formación de ⋃F',              '∀F∃u∀x(x∈u↔∃y(y∈F∧x∈y))', 'axioma', 'a∪b, sucesor S(x)'),
 ('5', 'Conjunto Potencia','Formación de ℘(a)',           '∀a∃p∀x(x∈p↔x⊆a)', 'axioma', 'Teorema de Cantor, producto, funciones'),
 ('6', 'Infinito',        'Existe un conjunto inductivo', '∃I[∅∈I ∧ ∀x(x∈I→x∪{x}∈I)]', 'axioma', 'ω, aritmética, inducción'),
 ('7', 'Separación',      'Subconjuntos definibles',      '∀a∃b∀x(x∈b↔x∈a∧φ(x))', 'ESQUEMA (∞ axiomas)', 'Evita Russell; ∩, ∖'),
 ('8', 'Reemplazo',       'Imagen de un conjunto es conjunto', '∀x∈a∃!yφ → ∃b∀y(y∈b↔∃x∈a φ)', 'ESQUEMA (∞ axiomas)', 'Ordinales grandes, recursión transfinita'),
 ('9', 'Regularidad',     'Todo conjunto ≠∅ tiene ∈-minimal', '∀x[x≠∅→∃y∈x(y∩x=∅)]', 'axioma', 'x∉x, rango, V=⋃V_α'),
 ('10','Elección',        'Función de elección',          '∀F[∅∉F → ∃f:F→⋃F ∀x∈F f(x)∈x]', 'axioma', 'Zorn, buen orden, Tychonoff'),
]
pd.set_option('display.max_colwidth', 80)
pd.DataFrame(filas, columns=['#', 'Axioma', 'Idea', 'Fórmula', 'Tipo', 'Consecuencias/uso']).set_index('#')

,Axioma,Idea,Fórmula,Tipo,Consecuencias/uso
#,,,,,
1,Extensionalidad,Igualdad de conjuntos,∀x∀y[∀z(z∈x↔z∈y) → x=y],axioma,"Unicidad de ∅, {a,b}, ⋃F, ℘(a)…"
2,Conjunto Vacío,Existencia de ∅,∃x∀y(y∉x),axioma (deducible de Separación),Base de la jerarquía
3,Par,"Formación de {a,b}",∀a∀b∃c∀x(x∈c↔x=a∨x=b),axioma,"Par ordenado, singleton"
4,Unión,Formación de ⋃F,∀F∃u∀x(x∈u↔∃y(y∈F∧x∈y)),axioma,"a∪b, sucesor S(x)"
5,Conjunto Potencia,Formación de ℘(a),∀a∃p∀x(x∈p↔x⊆a),axioma,"Teorema de Cantor, producto, funciones"
6,Infinito,Existe un conjunto inductivo,∃I[∅∈I ∧ ∀x(x∈I→x∪{x}∈I)],axioma,"ω, aritmética, inducción"
7,Separación,Subconjuntos definibles,∀a∃b∀x(x∈b↔x∈a∧φ(x)),ESQUEMA (∞ axiomas),"Evita Russell; ∩, ∖"
8,Reemplazo,Imagen de un conjunto es conjunto,∀x∈a∃!yφ → ∃b∀y(y∈b↔∃x∈a φ),ESQUEMA (∞ axiomas),"Ordinales grandes, recursión transfinita"
9,Regularidad,Todo conjunto ≠∅ tiene ∈-minimal,∀x[x≠∅→∃y∈x(y∩x=∅)],axioma,"x∉x, rango, V=⋃V_α"


### 11.2 Dependencias entre axiomas

| Hecho | Explicación |
|---|---|
| Vacío **se deduce** de Separación (+ existencia de algún conjunto) | $\{x\in a:x\neq x\}=\emptyset$. Se enuncia como axioma por tradición. |
| Par **se deduce** de Potencia + Reemplazo (o Potencia+Separación bajo cierta formulación) | $\mathcal P(\mathcal P(\emptyset))=\{\emptyset,\{\emptyset\}\}$, y reemplazando $\emptyset\mapsto a$, $\{\emptyset\}\mapsto b$ se obtiene $\{a,b\}$. |
| Separación **se deduce** de Reemplazo | Con la función “identidad condicionada” descrita en la sec. 8. |
| Extensionalidad, Unión, Potencia, Infinito, Reemplazo, Regularidad, Elección | No se deducen de los restantes (son **independientes**). |
| Regularidad **no** es necesaria para el análisis o el álgebra usuales | Basta con ZFC$^-$ (sin Fundación) o con $L$. |
| **Z** = ZFC sin Reemplazo | Menor fuerza: no prueba la existencia de $\aleph_\omega$. |
| **ZF** = ZFC sin Elección | Ver 11.3. |

### 11.3 Variantes y metateoría

- **Teoría de Zermelo (Z):** Extensionalidad, Vacío, Par, Unión, Potencia, Infinito, Separación. (Zermelo 1908.)
- **ZF:** $Z+$ Reemplazo + Regularidad. **ZFC:** ZF + Elección.
- **Aumentos de fuerza:** axiomas de **cardinales grandes** (inaccesibles, medibles, Woodin…) extienden ZFC y no son demostrables en ella.
- **Teoremas de Gödel.** Si ZFC es consistente, **no puede demostrar su propia consistencia** (2.º teorema de incompletitud), y existe un enunciado aritmético indecidible en ZFC (1.º teorema). Por tanto, la *consistencia de ZFC* es un acto de fe o se respalda con teorías más fuertes.
- **Hipótesis del Continuo (CH):** $2^{\aleph_0}=\aleph_1$. Gödel (1938) probó que es consistente con ZFC (en $L$) y Cohen (1963) que su negación también: **CH es independiente de ZFC**. Es el ejemplo más célebre de una cuestión matemática natural indecidible en ZFC.
- **Paradoja de Skolem:** si ZFC tiene modelo, tiene un modelo **numerable** (Löwenheim–Skolem), aunque en él se demuestre la existencia de conjuntos no numerables (la “no-numerabilidad” es relativa al modelo).
- **Clases propias:** $V$, $\mathrm{Ord}$, la clase de los grupos… no son conjuntos. Teorías alternativas como **NBG** o **Morse–Kelley** las incorporan formalmente; NBG es una extensión conservadora de ZFC.
- **Finitamente axiomatizables:** ZFC **no** lo es (Separación y Reemplazo son esquemas), mientras que NBG sí.

### 11.4 Construcción del universo: la jerarquía acumulativa

Reuniendo los axiomas, el universo “estándar” de ZFC se ve como

$$V_0=\emptyset,\qquad V_{\alpha+1}=\mathcal P(V_\alpha),\qquad V_\lambda=\bigcup_{\alpha<\lambda}V_\alpha\ (\lambda\text{ límite}),\qquad V=\bigcup_{\alpha\in\mathrm{Ord}}V_\alpha.$$

- Potencia genera los sucesores; Unión los límites; Infinito asegura que $\omega$ existe y $V_\omega\neq V$; Reemplazo permite iterar $\alpha$ tan lejos como se desee; Regularidad asegura que *todo* conjunto aparece en algún $V_\alpha$.
- $V_\omega$ = conjuntos hereditariamente finitos (el universo que se usó en este cuaderno); $V_\omega\models\mathrm{ZFC}-\mathrm{Infinito}$.

In [14]:
# Tamaños de los primeros niveles de la jerarquía acumulativa (tetración: 0,1,2,4,16,65536,...)
tam = [0, 1, 2, 4, 16, 65536]
print('|V_n| =', tam, '   (|V_{n+1}| = 2^{|V_n|})')
print('comprobación para n≤4:', [len(V_n(n)) for n in range(5)] == tam[:5])
print('V_5 tiene', 2**16, 'elementos; V_6 tendría 2**65536 (≈ 10^19728), ya inabarcable.')

# Regularidad + jerarquía: rango de los primeros niveles
print('rango(V_n) = n :', all(rango(V_n(n)) == n for n in range(5)))

|V_n| = [0, 1, 2, 4, 16, 65536]    (|V_{n+1}| = 2^{|V_n|})
comprobación para n≤4: True
V_5 tiene 65536 elementos; V_6 tendría 2**65536 (≈ 10^19728), ya inabarcable.
rango(V_n) = n : True


## 12. Bibliografía sugerida

- K. Kunen, *Set Theory* (2011), y *Set Theory: An Introduction to Independence Proofs* (1980). Referencia estándar para ZFC y forcing.
- T. Jech, *Set Theory*, 3.ª ed. (Springer, 2003). Enciclopédico; cubre cardinales grandes.
- H. B. Enderton, *Elements of Set Theory* (Academic Press, 1977). Introducción axiomática clara.
- P. Halmos, *Naive Set Theory* (1960). Introducción informal que sigue implícitamente ZFC.
- A. Levy, *Basic Set Theory* (Springer, 1979).
- E. Zermelo, *Untersuchungen über die Grundlagen der Mengenlehre I* (1908). Artículo original.
- K. Gödel, *The Consistency of the Axiom of Choice and of the Generalized Continuum-Hypothesis* (1938); P. Cohen, *The Independence of the Continuum Hypothesis* (1963).

> **Nota sobre el repositorio.** Este cuaderno complementa las unidades 2 (lógica de primer orden, cuantificadores) y 3 (conjuntos) del libro: los axiomas de este cuaderno son los que *justifican* las operaciones de conjuntos que allí se usan (unión, intersección, potencia, producto cartesiano).